# 08 — Multi-Agent Orchestration Patterns

**What you'll learn:**
- **Sequential pattern**: Agents in a pipeline, each building on the previous
- **Concurrent pattern**: Fan-out to multiple agents in parallel
- **Fan-in pattern**: Merge parallel outputs into one final decision or summary
- **Human-in-the-middle**: Add an approval gate before final actions
- When to use each pattern

---

## Two Core Multi-Agent Patterns

| Pattern | Flow | Use When |
|---------|------|----------|
| **Sequential** | A -> B -> C | Each step depends on the previous step's output |
| **Concurrent** | Fan-out to A, B, C in parallel -> merge | Steps are independent and can run simultaneously |

In [21]:
import asyncio
import os

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()


def make_client():
    return AzureOpenAIResponsesClient(
        project_endpoint=PROJECT_ENDPOINT,
        deployment_name=DEPLOYMENT_NAME,
        credential=credential,
    )

---

## Pattern 1: Sequential — Insurance Claims Processing

A claim goes through three agents in sequence:

```
Claim Narrative → [Intake Agent] → [Fraud Detection Agent] → [Adjuster Agent] → Final Decision
```

Each agent builds on the output of the previous one.

In [22]:
# Agent 1: Claims Intake — summarizes the raw narrative
intake_agent = make_client().as_agent(
    name="ClaimsIntakeAgent",
    instructions=(
        "You are a claims intake specialist. Given a raw claim narrative from a customer, "
        "produce a concise, structured summary including:\n"
        "- Claimant name and policy number\n"
        "- Date and type of incident\n"
        "- Estimated damages\n"
        "- Key facts relevant to the claim\n\n"
        "Keep your summary factual and under 150 words."
    ),
)

# Agent 2: Fraud Detection — reviews for red flags
fraud_agent = make_client().as_agent(
    name="FraudDetectionAgent",
    instructions=(
        "You are a fraud detection analyst. Given a claim summary, evaluate it for "
        "potential red flags. Look for:\n"
        "- Inconsistencies in the narrative\n"
        "- Unusually high claim amounts relative to the incident\n"
        "- Patterns common in fraudulent claims\n"
        "- Missing or vague details\n\n"
        "Provide a fraud risk rating (LOW / MEDIUM / HIGH) and list any concerns. "
        "Keep your analysis under 100 words."
    ),
)

# Agent 3: Claims Adjuster — makes the final determination
adjuster_agent = make_client().as_agent(
    name="ClaimsAdjusterAgent",
    instructions=(
        "You are a senior claims adjuster. Given a claim summary and a fraud analysis, "
        "make a final determination:\n"
        "- APPROVE: Pay the claim in full\n"
        "- PARTIAL APPROVE: Pay a reduced amount (specify why)\n"
        "- INVESTIGATE: Require further investigation before deciding\n"
        "- DENY: Reject the claim (specify reason)\n\n"
        "Include a 1-2 sentence rationale. Be decisive and professional."
    ),
)

print("Sequential agents created: Intake → Fraud Detection → Adjuster")

Sequential agents created: Intake → Fraud Detection → Adjuster


In [23]:
claim_narrative = """
My name is Thomas Reed, policy number POL-78234. On March 10, 2026, I was
driving home from work on Interstate 95 when another vehicle lost control on
wet pavement and sideswiped my car. The driver's side door is badly dented,
the mirror is broken, and there are deep scratches along the entire left side.
I also have some neck pain from the impact. The other driver admitted fault
at the scene. I got a police report (#PR-2026-4481). The body shop quoted
$7,200 for repairs and I've been to the doctor once so far — the visit was
$350. I need a rental car while mine is in the shop, estimated 2 weeks at
$45/day.
"""

# Step 1: Intake
print("=== Step 1: Claims Intake ===")
intake_result = await intake_agent.run(claim_narrative)
print(intake_result.text)
print()

=== Step 1: Claims Intake ===
**Claim Summary:**  
- **Claimant Name:** Thomas Reed  
- **Policy Number:** POL-78234  
- **Date of Incident:** March 10, 2026  
- **Type of Incident:** Vehicle collision (sideswipe on wet pavement)  
- **Estimated Damages:** $7,200 in vehicle repairs, $350 for medical visit, $630 for a 2-week rental car ($45/day)  
- **Key Facts:** Claimant’s car sustained damage to the driver’s side door, mirror, and had deep scratches along the left side. The other driver admitted fault at the scene. Police report (#PR-2026-4481) was obtained. Claimant reported neck pain and has seen a doctor once so far.



In [24]:
# Step 2: Fraud Detection — receives the intake summary
print("=== Step 2: Fraud Detection ===")
fraud_result = await fraud_agent.run(
    f"Review this claim summary for fraud indicators:\n\n{intake_result.text}"
)
print(fraud_result.text)
print()

=== Step 2: Fraud Detection ===
**Fraud Risk Rating:** LOW  

**Concerns:**  
- No significant red flags. The claim is detailed with specifics such as a police report, other driver admitting fault, and moderate, consistent costs.  
- Medical expenses and rental car costs seem reasonable and align with the reported incident.  
- Potential fraud patterns like exaggerated injuries or overly high repair estimates are not evident here.  
- Recommend verifying police report and medical records for routine due diligence.



In [25]:
# Step 3: Adjuster — receives both the summary and fraud analysis
print("=== Step 3: Claims Adjuster ===")
adjuster_result = await adjuster_agent.run(
    f"Make a final determination on this claim.\n\n"
    f"CLAIM SUMMARY:\n{intake_result.text}\n\n"
    f"FRAUD ANALYSIS:\n{fraud_result.text}"
)
print(adjuster_result.text)

=== Step 3: Claims Adjuster ===
**Final Determination:** APPROVE  

**Rationale:** The claim is well-documented with a police report and the other driver admitting fault, and no red flags were identified in the fraud analysis. The damages, medical costs, and rental car expenses are reasonable and consistent with the incident described. Routine verification of the police report and medical records is recommended, but they do not pose significant barriers to approval.


### Sequential Pattern Recap

```
Raw narrative
     │
     ▼
┌──────────────┐
│ Intake Agent  │  → structured summary
└──────┬───────┘
       │
       ▼
┌──────────────┐
│ Fraud Agent   │  → risk rating + concerns
└──────┬───────┘
       │
       ▼
┌──────────────┐
│ Adjuster Agent│  → APPROVE / PARTIAL / INVESTIGATE / DENY
└──────────────┘
```

Each agent is **specialized** — it does one thing well and passes its output
to the next agent in the chain.

---

## Pattern 2: Concurrent — Insurance Policy Comparison

A customer wants quotes for auto, home, and life insurance simultaneously.
We fan-out to three specialist agents in parallel, then merge the results.

```
Customer Profile ─┬─► [Auto Policy Agent]  ─┐
                  ├─► [Home Policy Agent]  ─┼─► Merged Comparison
                  └─► [Life Policy Agent]  ─┘
```

In [26]:
customer_profile = (
    "Customer: Maria Santos, age 38, married with 2 children.\n"
    "Location: Austin, TX.\n"
    "Vehicles: 2022 Honda CR-V, 2020 Toyota Camry.\n"
    "Home: 3-bed/2-bath single-family, built 2005, valued at $420K.\n"
    "Income: $110K/year combined household.\n"
    "Clean driving record, no prior claims."
)

# Create three specialist agents
auto_agent = make_client().as_agent(
    name="AutoPolicyAgent",
    instructions=(
        "You are an auto insurance specialist. Given a customer profile, provide:\n"
        "1. Recommended coverage (liability limits, collision, comprehensive)\n"
        "2. Estimated monthly premium range\n"
        "3. Available discounts\n"
        "Keep your response under 100 words."
    ),
)

home_agent = make_client().as_agent(
    name="HomePolicyAgent",
    instructions=(
        "You are a homeowners insurance specialist. Given a customer profile, provide:\n"
        "1. Recommended coverage (dwelling, personal property, liability)\n"
        "2. Estimated monthly premium range\n"
        "3. Available discounts or bundling options\n"
        "Keep your response under 100 words."
    ),
)

life_agent = make_client().as_agent(
    name="LifePolicyAgent",
    instructions=(
        "You are a life insurance specialist. Given a customer profile, provide:\n"
        "1. Recommended policy type (term vs whole) and coverage amount\n"
        "2. Estimated monthly premium range\n"
        "3. Key considerations for this customer\n"
        "Keep your response under 100 words."
    ),
)

print("Concurrent agents created: Auto, Home, Life")

Concurrent agents created: Auto, Home, Life


In [27]:
# Fan-out: Run all three agents concurrently
print("=== Running 3 agents concurrently... ===\n")

auto_task = auto_agent.run(f"Provide an auto insurance quote for:\n{customer_profile}")
home_task = home_agent.run(f"Provide a homeowners insurance quote for:\n{customer_profile}")
life_task = life_agent.run(f"Provide a life insurance quote for:\n{customer_profile}")

# asyncio.gather runs all three in parallel
auto_result, home_result, life_result = await asyncio.gather(
    auto_task, home_task, life_task
)

print("=== AUTO INSURANCE ===")
print(auto_result.text)
print()
print("=== HOME INSURANCE ===")
print(home_result.text)
print()
print("=== LIFE INSURANCE ===")
print(life_result.text)

=== Running 3 agents concurrently... ===

=== AUTO INSURANCE ===
1. **Recommended Coverage**:  
   - Liability: 100/300/100  
   - Collision & Comprehensive: $500 deductible  
   - Uninsured/underinsured motorist: Match liability limits  

2. **Estimated Monthly Premium**: $110–$150  

3. **Available Discounts**:  
   - Multi-car  
   - Bundling auto and home insurance  
   - Good driver  
   - Married/family  
   - Low annual mileage (if applicable)  

=== HOME INSURANCE ===
1. Recommended Coverage:
   - Dwelling: $420K replacement cost
   - Personal Property: $210K
   - Liability: $500K
   - Additional Living Expenses: $84K

2. Estimated Monthly Premium: $120-$180

3. Discounts/Bundling:
   - Multi-policy discount (home + auto insurance)
   - Clean record discount
   - Home security system discount
   - Loyalty or new customer discounts if applicable. 

Bundle coverage for potential savings of 15-25%.

=== LIFE INSURANCE ===
1. **Policy Type and Coverage:** Term life insurance, $500,

### Merge: Create a Unified Comparison

A summary agent takes all three quotes and produces a comparison.

In [28]:
summary_agent = make_client().as_agent(
    name="ComparisonAgent",
    instructions=(
        "You are an insurance advisor. Given quotes from three specialists (auto, home, life), "
        "create a concise comparison summary for the customer. Include:\n"
        "1. Total estimated monthly cost across all policies\n"
        "2. Bundle discount opportunities\n"
        "3. Your top recommendation\n"
        "Keep it under 150 words and format it nicely."
    ),
)

comparison = await summary_agent.run(
    f"Create a comparison summary for Maria Santos.\n\n"
    f"AUTO QUOTE:\n{auto_result.text}\n\n"
    f"HOME QUOTE:\n{home_result.text}\n\n"
    f"LIFE QUOTE:\n{life_result.text}"
)

print("=== UNIFIED COMPARISON ===")
print(comparison.text)

=== UNIFIED COMPARISON ===
### Summary for Maria Santos  

**Total Estimated Monthly Cost:**  
- Auto: $110–$150  
- Home: $120–$180  
- Life: $25–$40  
**Estimated Total:** $255–$370  

**Bundle Discount Opportunities:**  
- Save 15–25% by bundling auto and home insurance, potentially reducing costs by $65–$90 monthly.  
- Additional discounts for good driver status, low annual mileage, home security systems, and family status.

**Recommendation:**  
Bundle auto and home insurance with one provider for maximum savings. Select life insurance coverage that balances affordability and long-term protection for your family. Prioritize reviewing bundled quotes for a clearer picture of total savings.


### Concurrent Pattern Recap

```
Customer Profile
     │
     ├──► Auto Agent  ──┐
     │                  │
     ├──► Home Agent  ──┼──► Summary Agent ──► Comparison
     │                  │
     └──► Life Agent  ──┘
```

- `asyncio.gather()` runs all agents in parallel — 3x faster than sequential
- A final summary agent merges the results
- Each specialist is independent — no shared state

---

## Choosing the Right Pattern

| Question | Sequential | Concurrent |
|----------|-----------|------------|
| Does step B need output from step A? | ✅ Yes | ❌ No |
| Are the steps independent? | ❌ No | ✅ Yes |
| Is latency critical? | Slower | Faster |
| Example | Claims processing pipeline | Multi-product quoting |

You can also **combine** them — e.g., run fraud detection and compliance check
concurrently, then sequentially feed both into an adjuster.

In [29]:
await credential.close()
print("Done!")

Done!


---

## Key Takeaways

- **Sequential**: Chain agents where each step feeds the next - claims intake -> fraud -> adjuster
- **Concurrent**: `asyncio.gather()` runs independent agents in parallel - auto + home + life quotes
- **Fan-out/Fan-in**: Parallel specialist execution can be merged into one final summary or decision
- **Human-in-the-middle**: Add policy-driven approval gates before executing sensitive outcomes
- These patterns compose - sequential stages can contain concurrent sub-steps and review checkpoints
- For explicit graph-based control, combine these patterns with **Workflows** (notebook 07)

---

**Next up:** [08 — MCP with Agents](./09-mcp-with-agents.ipynb) —
combine agents with workflows for sequential claims processing and concurrent policy quoting.